# Lab 07: Transfer Learning for Crack Detection (Ali Hamza)

**Course:** COMP-341L - Artificial Neural Networks Lab  
**Student:** Ali Hamza  
**Roll Number:** B23F0063AI106  
**Section:** B.S AI - Red  
**Execution Environment:** Google Colab

This notebook follows the Lab 07 manual step by step:
1. CNN from scratch (overfitting baseline)
2. Transfer learning with frozen base (MobileNetV2)
3. Data augmentation experiment
4. Fine-tuning last layers
5. Padding analysis (`valid` vs `same`)
6. Final evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)


In [ ]:
# Optional: Mount Google Drive for persistent saving in Colab.
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from datetime import datetime
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (128, 128)
BATCH_SIZE = 32
BASELINE_EPOCHS = 15
TRANSFER_EPOCHS = 12
FINETUNE_EPOCHS = 5
AUTOTUNE = tf.data.AUTOTUNE

STUDENT_NAME = 'Ali Hamza'
ROLL_NUMBER = 'B23F0063AI106'
SECTION = 'B.S AI - Red'

# Set this in Colab to store outputs on Drive if needed.
# Example: /content/drive/MyDrive/COMP-341L/Lab 7/Ali Hamza's Lab
BASE_DIR = os.environ.get('LAB7_BASE_DIR', '.')
PLOTS_DIR = os.path.join(BASE_DIR, 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

print('TensorFlow version:', tf.__version__)
print('BASE_DIR:', os.path.abspath(BASE_DIR))
print('PLOTS_DIR:', os.path.abspath(PLOTS_DIR))


## Dataset Setup (Small-Data Scenario: 1,200 images)

Lab allows CIFAR-10 simulation for binary crack detection.  
We use two CIFAR-10 classes and map them to:
- `Crack Present` (1)
- `No Crack` (0)

Then we create a **small balanced dataset of 1,200 images** to match the real-world constraint.


In [ ]:
# Load CIFAR-10
(x_train_full, y_train_full), (x_test_full, y_test_full) = tf.keras.datasets.cifar10.load_data()

x_all = np.concatenate([x_train_full, x_test_full], axis=0)
y_all = np.concatenate([y_train_full, y_test_full], axis=0).squeeze()

# Simulated binary classes from CIFAR-10
# class 3 (cat)    -> Crack Present (1)
# class 5 (dog)    -> No Crack (0)
CLASS_CRACK = 3
CLASS_NO_CRACK = 5

mask = np.isin(y_all, [CLASS_CRACK, CLASS_NO_CRACK])
x_bin = x_all[mask]
y_bin_raw = y_all[mask]
y_bin = (y_bin_raw == CLASS_CRACK).astype(np.float32)

# Small dataset: 600 from each class = 1,200 total
rng = np.random.default_rng(SEED)
idx_crack = np.where(y_bin == 1)[0]
idx_no_crack = np.where(y_bin == 0)[0]

selected = np.concatenate([
    rng.choice(idx_crack, size=600, replace=False),
    rng.choice(idx_no_crack, size=600, replace=False),
])
rng.shuffle(selected)

x_small = x_bin[selected]
y_small = y_bin[selected]

print('Small dataset shape:', x_small.shape)
print('Label distribution:', {
    'Crack Present (1)': int((y_small == 1).sum()),
    'No Crack (0)': int((y_small == 0).sum())
})


In [ ]:
# Train/Validation/Test split: 70% / 15% / 15%
n_total = len(x_small)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)

x_train = x_small[:n_train]
y_train = y_small[:n_train]

x_val = x_small[n_train:n_train + n_val]
y_val = y_small[n_train:n_train + n_val]

x_test = x_small[n_train + n_val:]
y_test = y_small[n_train + n_val:]

print('Train:', x_train.shape, y_train.shape)
print('Val  :', x_val.shape, y_val.shape)
print('Test :', x_test.shape, y_test.shape)


In [ ]:
# Visualize sample images
label_names = {0: 'No Crack', 1: 'Crack Present'}

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(x_train[i])
    ax.set_title(label_names[int(y_train[i])])
    ax.axis('off')

plt.suptitle('Sample Images (Simulated Crack vs No Crack)')
plt.tight_layout()
sample_path = os.path.join(PLOTS_DIR, 'task0_sample_images.png')
plt.savefig(sample_path, dpi=130, bbox_inches='tight')
plt.show()
print('Saved:', sample_path)


In [ ]:
def preprocess_image(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, IMG_SIZE)
    label = tf.cast(label, tf.float32)
    return image, label


def make_dataset(x, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(buffer_size=len(x), seed=SEED)
    ds = ds.map(preprocess_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(x_train, y_train, training=True)
val_ds = make_dataset(x_val, y_val, training=False)
test_ds = make_dataset(x_test, y_test, training=False)


## Part 1 - Baseline CNN From Scratch (Failure Demonstration)
Architecture required by lab:
- Conv -> ReLU -> MaxPool
- Conv -> ReLU -> MaxPool
- Dense -> Softmax/Sigmoid

We use binary sigmoid output and train for **15 epochs**.


In [ ]:
def build_scratch_cnn(input_shape=(128, 128, 3)):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),
        tf.keras.layers.Rescaling(1./255),

        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D(2),

        tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling2D(2),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(1, activation='sigmoid'),
    ], name='scratch_cnn')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model


scratch_model = build_scratch_cnn()
scratch_model.summary()


In [ ]:
history_scratch = scratch_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=BASELINE_EPOCHS,
    verbose=1,
)


In [ ]:
scratch_train_acc = history_scratch.history['accuracy'][-1]
scratch_val_acc = history_scratch.history['val_accuracy'][-1]
scratch_gap = scratch_train_acc - scratch_val_acc

print(f'Baseline Train Accuracy : {scratch_train_acc:.4f}')
print(f'Baseline Val Accuracy   : {scratch_val_acc:.4f}')
print(f'Overfitting Gap         : {scratch_gap:.4f}')


### Reflection (Part 1)
- Overfitting happens because the model has enough capacity to memorize a small dataset (1,200 images).
- With limited examples and image variation, training accuracy rises while validation lags.
- Small dataset size reduces coverage of lighting, rotation, blur, and zoom conditions, so generalization suffers.


## Part 2 - Transfer Learning (Feature Extraction)
### Step 1: Load Pretrained Base
Using **MobileNetV2** with ImageNet weights.

### Step 2: Freeze Base
`base_model.trainable = False`

Mathematically, for frozen parameters `theta_base`:
- `theta_base(t+1) = theta_base(t)` (no update)
- Gradients may be computed through the graph, but optimizer applies updates only to trainable variables.

### Step 3: Add Custom Head
- GlobalAveragePooling2D
- Dense(128, ReLU)
- Dropout(0.5)
- Dense(1, sigmoid)


In [ ]:
def build_transfer_model(base_model, use_augmentation=False):
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.12),
        tf.keras.layers.RandomZoom(0.10),
        tf.keras.layers.RandomContrast(0.10),
    ], name='data_augmentation')

    inputs = tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
    x = inputs

    if use_augmentation:
        x = data_augmentation(x)

    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

    # Keep BatchNorm behavior stable by calling base with training=False
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs, outputs, name='mobilenetv2_transfer')
    return model


base_model_frozen = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet'
)
base_model_frozen.trainable = False

transfer_frozen = build_transfer_model(base_model_frozen, use_augmentation=False)
transfer_frozen.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
transfer_frozen.summary()


In [ ]:
history_transfer_frozen = transfer_frozen.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TRANSFER_EPOCHS,
    verbose=1,
)

frozen_train_acc = history_transfer_frozen.history['accuracy'][-1]
frozen_val_acc = history_transfer_frozen.history['val_accuracy'][-1]
frozen_gap = frozen_train_acc - frozen_val_acc

print(f'Frozen Transfer Train Accuracy : {frozen_train_acc:.4f}')
print(f'Frozen Transfer Val Accuracy   : {frozen_val_acc:.4f}')
print(f'Overfitting Gap                : {frozen_gap:.4f}')


## Part 3 - Data Augmentation Experiment
Now we repeat frozen transfer learning but enable augmentation:
- RandomRotation
- RandomZoom
- RandomFlip
- RandomContrast


In [ ]:
base_model_aug = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet'
)
base_model_aug.trainable = False

transfer_aug = build_transfer_model(base_model_aug, use_augmentation=True)
transfer_aug.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_transfer_aug = transfer_aug.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TRANSFER_EPOCHS,
    verbose=1,
)

aug_train_acc = history_transfer_aug.history['accuracy'][-1]
aug_val_acc = history_transfer_aug.history['val_accuracy'][-1]
aug_gap = aug_train_acc - aug_val_acc

print(f'Frozen+Aug Train Accuracy : {aug_train_acc:.4f}')
print(f'Frozen+Aug Val Accuracy   : {aug_val_acc:.4f}')
print(f'Overfitting Gap           : {aug_gap:.4f}')
print(f'Validation change vs no-aug: {aug_val_acc - frozen_val_acc:+.4f}')


**Why augmentation helps:**
It synthetically increases diversity (rotation, zoom, contrast, flips), forcing the model to learn robust patterns rather than memorizing fixed image layouts.


## Part 4 - Fine-Tuning (Unfreeze Last 20 Layers)
- Unfreeze base model
- Freeze all but last 20 layers
- Reduce learning rate to `1e-5`
- Train 5 more epochs

**Why reduce learning rate?**
Because pretrained weights are already useful; large updates can destroy learned features (catastrophic forgetting) and destabilize validation performance.


In [ ]:
base_model_aug.trainable = True
for layer in base_model_aug.layers[:-20]:
    layer.trainable = False

# Keep BatchNorm frozen for stable fine-tuning on small data
for layer in base_model_aug.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

transfer_aug.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_finetune = transfer_aug.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINETUNE_EPOCHS,
    verbose=1,
)


In [ ]:
def combine_histories(h1, h2):
    merged = {}
    for key in h1.history.keys():
        merged[key] = h1.history[key] + h2.history[key]
    return merged

history_aug_finetuned = combine_histories(history_transfer_aug, history_finetune)

finetune_train_acc = history_aug_finetuned['accuracy'][-1]
finetune_val_acc = history_aug_finetuned['val_accuracy'][-1]
finetune_gap = finetune_train_acc - finetune_val_acc

print(f'Fine-Tuned Train Accuracy : {finetune_train_acc:.4f}')
print(f'Fine-Tuned Val Accuracy   : {finetune_val_acc:.4f}')
print(f'Overfitting Gap           : {finetune_gap:.4f}')
print(f'Validation change vs Frozen+Aug: {finetune_val_acc - aug_val_acc:+.4f}')


## Comparison Table (Required)
We compare:
- CNN Scratch
- Transfer (Frozen)
- Transfer (Fine-Tuned)


In [ ]:
comparison_df = pd.DataFrame([
    {
        'Model': 'CNN Scratch',
        'Train Acc': scratch_train_acc,
        'Val Acc': scratch_val_acc,
        'Overfitting Gap': scratch_gap,
        'Overfitting?': 'Yes' if scratch_gap > 0.05 else 'Low/No'
    },
    {
        'Model': 'Transfer (Frozen)',
        'Train Acc': frozen_train_acc,
        'Val Acc': frozen_val_acc,
        'Overfitting Gap': frozen_gap,
        'Overfitting?': 'Yes' if frozen_gap > 0.05 else 'Low/No'
    },
    {
        'Model': 'Transfer (Fine-Tuned)',
        'Train Acc': finetune_train_acc,
        'Val Acc': finetune_val_acc,
        'Overfitting Gap': finetune_gap,
        'Overfitting?': 'Yes' if finetune_gap > 0.05 else 'Low/No'
    },
])

comparison_df


## Part 5 - Padding Analysis (`valid` vs `same`)
We build two small CNNs and compare output shapes.


In [ ]:
report_md = f"""# Lab Report 7: Transfer Learning for Crack Detection

**Course Code:** COMP-341L  
**Course Name:** Artificial Neural Networks Lab  
**Lab Number:** 7  
**Date:** {datetime.now().strftime('%B %d, %Y')}  
**Name:** {STUDENT_NAME}  
**Roll Number:** {ROLL_NUMBER}  
**Section:** {SECTION}

## Objective
Build a robust binary image classifier (Crack Present vs No Crack) under small-data constraints using transfer learning.

## Part 1: CNN from Scratch (15 epochs)
- Train Accuracy: {scratch_train_acc:.4f}
- Validation Accuracy: {scratch_val_acc:.4f}
- Overfitting Gap: {scratch_gap:.4f}

## Part 2: Transfer Learning (Frozen MobileNetV2)
- Train Accuracy: {frozen_train_acc:.4f}
- Validation Accuracy: {frozen_val_acc:.4f}
- Overfitting Gap: {frozen_gap:.4f}

## Part 3 and 4: Augmentation + Fine-Tuning
- Fine-Tuned Train Accuracy: {finetune_train_acc:.4f}
- Fine-Tuned Validation Accuracy: {finetune_val_acc:.4f}
- Fine-Tuned Overfitting Gap: {finetune_gap:.4f}

## Final Test Metrics (Fine-Tuned Model)
- Accuracy: {acc:.4f}
- Precision: {prec:.4f}
- Recall: {rec:.4f}
- F1-score: {f1:.4f}

## Comparison Table
{comparison_df.to_string(index=False)}

## Key Observations
- Transfer learning reduced overfitting compared to scratch baseline.
- Data augmentation improved validation robustness.
- Fine-tuning with small learning rate improved adaptation while preserving pretrained features.
- High recall is critical because missing cracks can create severe safety risk.

## Plots
- ![Sample Images](plots/task0_sample_images.png)
- ![Accuracy Curves](plots/task_visual_accuracy_curves.png)
- ![Loss Curves](plots/task_visual_loss_curves.png)
- ![Overfitting Comparison](plots/task_visual_overfitting_comparison.png)
- ![Confusion Matrix](plots/task_final_confusion_matrix.png)
"""

report_html = f"""<!doctype html>
<html lang='en'>
<head>
  <meta charset='utf-8'>
  <title>Lab Report 7 - {STUDENT_NAME}</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 28px; line-height: 1.5; color: #111; }}
    h1, h2, h3 {{ margin-top: 24px; }}
    img {{ max-width: 900px; width: 100%; border: 1px solid #ddd; margin: 8px 0 18px; }}
    table {{ border-collapse: collapse; width: 100%; margin: 12px 0; }}
    th, td {{ border: 1px solid #ccc; padding: 8px; text-align: left; }}
    th {{ background: #f4f4f4; }}
  </style>
</head>
<body>
  <h1>Lab Report 7: Transfer Learning for Crack Detection</h1>
  <p><strong>Course Code:</strong> COMP-341L<br>
  <strong>Date:</strong> {datetime.now().strftime('%B %d, %Y')}<br>
  <strong>Name:</strong> {STUDENT_NAME}<br>
  <strong>Roll Number:</strong> {ROLL_NUMBER}<br>
  <strong>Section:</strong> {SECTION}</p>

  <h2>Final Test Metrics (Fine-Tuned Model)</h2>
  <ul>
    <li>Accuracy: {acc:.4f}</li>
    <li>Precision: {prec:.4f}</li>
    <li>Recall: {rec:.4f}</li>
    <li>F1-score: {f1:.4f}</li>
  </ul>

  <h2>Required Plots</h2>
  <img src='plots/task_visual_accuracy_curves.png' alt='Accuracy curves'>
  <img src='plots/task_visual_loss_curves.png' alt='Loss curves'>
  <img src='plots/task_visual_overfitting_comparison.png' alt='Overfitting comparison'>
  <img src='plots/task_final_confusion_matrix.png' alt='Confusion matrix'>

  <h2>Conclusion</h2>
  <p>Transfer learning with augmentation and controlled fine-tuning improved validation and test behavior versus scratch CNN on small data.</p>
</body>
</html>
"""

md_path = os.path.join(BASE_DIR, 'Lab_Report_7.md')
html_path = os.path.join(BASE_DIR, 'Lab_Report_7.html')

with open(md_path, 'w', encoding='utf-8') as f:
    f.write(report_md)
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(report_html)

comparison_path = os.path.join(PLOTS_DIR, 'task_comparison_table.csv')
comparison_df.to_csv(comparison_path, index=False)

print('Saved:', os.path.abspath(md_path))
print('Saved:', os.path.abspath(html_path))
print('Saved:', os.path.abspath(comparison_path))

print('')
print('Files in plots:')
for fn in sorted(os.listdir(PLOTS_DIR)):
    print(' -', fn)


**Why border information matters for crack detection:**
Cracks often appear near image edges. `valid` padding shrinks spatial maps and can drop border evidence early, while `same` padding preserves edge context across layers.


## Required Visualizations
1. Training vs validation accuracy  
2. Training vs validation loss  
3. Overfitting comparison graph


In [ ]:
def plot_acc_loss(histories, title_prefix, acc_path, loss_path):
    plt.figure(figsize=(10, 5))
    for name, hist in histories.items():
        plt.plot(hist['accuracy'], label=f'{name} Train')
        plt.plot(hist['val_accuracy'], linestyle='--', label=f'{name} Val')
    plt.title(f'{title_prefix} - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(acc_path, dpi=130, bbox_inches='tight')
    plt.show()

    plt.figure(figsize=(10, 5))
    for name, hist in histories.items():
        plt.plot(hist['loss'], label=f'{name} Train')
        plt.plot(hist['val_loss'], linestyle='--', label=f'{name} Val')
    plt.title(f'{title_prefix} - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(loss_path, dpi=130, bbox_inches='tight')
    plt.show()


histories_for_plot = {
    'Scratch': history_scratch.history,
    'Frozen TL': history_transfer_frozen.history,
    'Fine-Tuned TL': history_aug_finetuned,
}

acc_curve_path = os.path.join(PLOTS_DIR, 'task_visual_accuracy_curves.png')
loss_curve_path = os.path.join(PLOTS_DIR, 'task_visual_loss_curves.png')
plot_acc_loss(histories_for_plot, 'Training vs Validation', acc_curve_path, loss_curve_path)

print('Saved:', acc_curve_path)
print('Saved:', loss_curve_path)

# Overfitting comparison graph
model_names = comparison_df['Model'].tolist()
gaps = comparison_df['Overfitting Gap'].tolist()

plt.figure(figsize=(8, 5))
colors = ['#d9534f', '#5bc0de', '#5cb85c']
plt.bar(model_names, gaps, color=colors)
plt.axhline(0.05, color='black', linestyle='--', linewidth=1, label='Gap=0.05')
plt.title('Overfitting Gap Comparison')
plt.ylabel('Train Acc - Val Acc')
plt.xticks(rotation=15)
plt.legend()
plt.tight_layout()
overfit_path = os.path.join(PLOTS_DIR, 'task_visual_overfitting_comparison.png')
plt.savefig(overfit_path, dpi=130, bbox_inches='tight')
plt.show()
print('Saved:', overfit_path)


## Final Evaluation (Best Model on Test Set)
We evaluate the fine-tuned transfer model with:
- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix


In [ ]:
# Use fine-tuned model (transfer_aug after fine-tuning)
y_true = []
y_prob = []

for xb, yb in test_ds:
    probs = transfer_aug.predict(xb, verbose=0).squeeze()
    y_prob.extend(np.atleast_1d(probs).tolist())
    y_true.extend(yb.numpy().astype(int).tolist())

y_true = np.array(y_true)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f'Accuracy : {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'F1-score : {f1:.4f}')

print('')
print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=['No Crack', 'Crack Present'], digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix (Fine-Tuned Model)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks([0, 1], ['No Crack', 'Crack Present'])
plt.yticks([0, 1], ['No Crack', 'Crack Present'])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        plt.text(j, i, str(cm[i, j]), ha='center', va='center', color=color)

plt.tight_layout()
cm_path = os.path.join(PLOTS_DIR, 'task_final_confusion_matrix.png')
plt.savefig(cm_path, dpi=130, bbox_inches='tight')
plt.show()
print('Saved:', cm_path)


### Why Recall is Important in Crack Detection
Recall measures how many actual cracks are correctly detected. In safety-critical inspection, a false negative (missing a real crack) is dangerous, so high recall is essential.


## Export Lab Report Files (`Lab_Report_7.md` and `Lab_Report_7.html`)
Run next cell after training to generate report files with your actual results.


In [ ]:
report_md = f"""# Lab Report 7: Transfer Learning for Crack Detection

**Course Code:** COMP-341L  
**Course Name:** Artificial Neural Networks Lab  
**Lab Number:** 7  
**Date:** {datetime.now().strftime('%B %d, %Y')}  
**Name:** {STUDENT_NAME}  
**Roll Number:** {ROLL_NUMBER}  
**Section:** {SECTION}

## Objective
Build a robust binary image classifier (Crack Present vs No Crack) under small-data constraints using transfer learning.

## Part 1: CNN from Scratch (15 epochs)
- Train Accuracy: {scratch_train_acc:.4f}
- Validation Accuracy: {scratch_val_acc:.4f}
- Overfitting Gap: {scratch_gap:.4f}

## Part 2: Transfer Learning (Frozen MobileNetV2)
- Train Accuracy: {frozen_train_acc:.4f}
- Validation Accuracy: {frozen_val_acc:.4f}
- Overfitting Gap: {frozen_gap:.4f}

## Part 3 and 4: Augmentation + Fine-Tuning
- Fine-Tuned Train Accuracy: {finetune_train_acc:.4f}
- Fine-Tuned Validation Accuracy: {finetune_val_acc:.4f}
- Fine-Tuned Overfitting Gap: {finetune_gap:.4f}

## Final Test Metrics (Fine-Tuned Model)
- Accuracy: {acc:.4f}
- Precision: {prec:.4f}
- Recall: {rec:.4f}
- F1-score: {f1:.4f}

## Comparison Table
{comparison_df.to_string(index=False)}

## Key Observations
- Transfer learning reduced overfitting compared to scratch baseline.
- Data augmentation improved validation robustness.
- Fine-tuning with small learning rate improved adaptation while preserving pretrained features.
- High recall is critical because missing cracks can create severe safety risk.

## Plots
- ![Sample Images](plots/task0_sample_images.png)
- ![Accuracy Curves](plots/task_visual_accuracy_curves.png)
- ![Loss Curves](plots/task_visual_loss_curves.png)
- ![Overfitting Comparison](plots/task_visual_overfitting_comparison.png)
- ![Confusion Matrix](plots/task_final_confusion_matrix.png)
"""

report_html = f"""<!doctype html>
<html lang='en'>
<head>
  <meta charset='utf-8'>
  <title>Lab Report 7 - {STUDENT_NAME}</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 28px; line-height: 1.5; color: #111; }}
    h1, h2, h3 {{ margin-top: 24px; }}
    img {{ max-width: 900px; width: 100%; border: 1px solid #ddd; margin: 8px 0 18px; }}
    table {{ border-collapse: collapse; width: 100%; margin: 12px 0; }}
    th, td {{ border: 1px solid #ccc; padding: 8px; text-align: left; }}
    th {{ background: #f4f4f4; }}
  </style>
</head>
<body>
  <h1>Lab Report 7: Transfer Learning for Crack Detection</h1>
  <p><strong>Course Code:</strong> COMP-341L<br>
  <strong>Date:</strong> {datetime.now().strftime('%B %d, %Y')}<br>
  <strong>Name:</strong> {STUDENT_NAME}<br>
  <strong>Roll Number:</strong> {ROLL_NUMBER}<br>
  <strong>Section:</strong> {SECTION}</p>

  <h2>Final Test Metrics (Fine-Tuned Model)</h2>
  <ul>
    <li>Accuracy: {acc:.4f}</li>
    <li>Precision: {prec:.4f}</li>
    <li>Recall: {rec:.4f}</li>
    <li>F1-score: {f1:.4f}</li>
  </ul>

  <h2>Required Plots</h2>
  <img src='plots/task_visual_accuracy_curves.png' alt='Accuracy curves'>
  <img src='plots/task_visual_loss_curves.png' alt='Loss curves'>
  <img src='plots/task_visual_overfitting_comparison.png' alt='Overfitting comparison'>
  <img src='plots/task_final_confusion_matrix.png' alt='Confusion matrix'>

  <h2>Conclusion</h2>
  <p>Transfer learning with augmentation and controlled fine-tuning improved validation and test behavior versus scratch CNN on small data.</p>
</body>
</html>
"""

md_path = os.path.join(BASE_DIR, 'Lab_Report_7.md')
html_path = os.path.join(BASE_DIR, 'Lab_Report_7.html')

with open(md_path, 'w', encoding='utf-8') as f:
    f.write(report_md)
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(report_html)

comparison_path = os.path.join(PLOTS_DIR, 'task_comparison_table.csv')
comparison_df.to_csv(comparison_path, index=False)

print('Saved:', os.path.abspath(md_path))
print('Saved:', os.path.abspath(html_path))
print('Saved:', os.path.abspath(comparison_path))

print('')
print('Files in plots:')
for fn in sorted(os.listdir(PLOTS_DIR)):
    print(' -', fn)


In [ ]:
# Optional: Zip deliverables for quick download from Colab.
# import shutil
# shutil.make_archive('Lab_7_Deliverables', 'zip', BASE_DIR)
# print('Created Lab_7_Deliverables.zip')
